# データ分析ひな型
CSVを読み込んでから以下のセルを順番に実行してください。

In [1]:
# Cell 1: CSVの読み込み（ここだけ編集）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'MS Gothic'  # Windows日本語フォント

CSV_PATH = r"C:\Users\RD004\Documents\lab\data\iemocap\IEMOCAP正解ラベル.csv"
df = pd.read_csv(CSV_PATH)
print("読み込み完了")

読み込み完了


In [2]:
# Cell 2: 基本情報
print("=== 行数・列数 ===")
print(f"行数: {len(df)}, 列数: {len(df.columns)}")

print("\n=== 列名と型 ===")
print(df.dtypes)

print("\n=== 先頭5行 ===")
df.head()

=== 行数・列数 ===
行数: 10039, 列数: 11

=== 列名と型 ===
filename        object
ID              object
Session         object
SubSession      object
dialog          object
spontanious     object
emo             object
sex             object
Valence        float64
Arousal        float64
Dominance      float64
dtype: object

=== 先頭5行 ===


,filename,ID,Session,SubSession,dialog,spontanious,emo,sex,Valence,Arousal,Dominance
0,Ses01F_impro01_F000.wav,Ses01F_impro01_F000,Ses01,Ses01F,impro01,impro,neu,F,2.5,2.5,2.5
1,Ses01F_impro01_F001.wav,Ses01F_impro01_F001,Ses01,Ses01F,impro01,impro,neu,F,2.5,2.5,2.5
2,Ses01F_impro01_F002.wav,Ses01F_impro01_F002,Ses01,Ses01F,impro01,impro,neu,F,2.5,2.5,2.5
3,Ses01F_impro01_F003.wav,Ses01F_impro01_F003,Ses01,Ses01F,impro01,impro,xxx,F,2.5,3.0,3.0
4,Ses01F_impro01_F004.wav,Ses01F_impro01_F004,Ses01,Ses01F,impro01,impro,xxx,F,2.5,3.0,2.5


In [3]:
# Cell 3: 欠損値の確認
print("=== 欠損値の数 ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "欠損値なし")

print("\n=== ユニーク値の数（列ごと） ===")
print(df.nunique())

=== 欠損値の数 ===
欠損値なし

=== ユニーク値の数（列ごと） ===
filename       10039
ID             10039
Session            5
SubSession        10
dialog            15
spontanious        2
emo               11
sex                2
Valence           21
Arousal           22
Dominance         21
dtype: int64


In [4]:
# Cell 4: 各列のユニーク値を表示（カテゴリ列の確認）
for col in df.columns:
    n = df[col].nunique()
    if n <= 30:  # ユニーク値が少ない列はすべて表示
        print(f"\n[{col}] ({n}種類)")
        print(df[col].value_counts().to_string())
    else:
        print(f"\n[{col}] ユニーク値が多い ({n}種類) — 先頭5件: {df[col].head().tolist()}")


[filename] ユニーク値が多い (10039種類) — 先頭5件: ['Ses01F_impro01_F000.wav', 'Ses01F_impro01_F001.wav', 'Ses01F_impro01_F002.wav', 'Ses01F_impro01_F003.wav', 'Ses01F_impro01_F004.wav']

[ID] ユニーク値が多い (10039種類) — 先頭5件: ['Ses01F_impro01_F000', 'Ses01F_impro01_F001', 'Ses01F_impro01_F002', 'Ses01F_impro01_F003', 'Ses01F_impro01_F004']

[Session] (5種類)
Session
Ses05    2170
Ses03    2136
Ses04    2103
Ses01    1819
Ses02    1811

[SubSession] (10種類)
SubSession
Ses03M    1178
Ses05F    1128
Ses04F    1105
Ses05M    1042
Ses04M     998
Ses01M     958
Ses03F     958
Ses02M     922
Ses02F     889
Ses01F     861

[dialog] (15種類)
dialog
script01    2087
script03    1597
script02    1571
impro07      789
impro03      713
impro04      610
impro02      558
impro05      554
impro01      464
impro06      447
impro08      411
impro05b      66
impro08a      64
impro05a      60
impro08b      48

[spontanious] (2種類)
spontanious
script    5255
impro     4784

[emo] (11種類)
emo
xxx    2507
fru    1849
neu    1708
ang

In [5]:
# Cell 5: 数値列の基本統計量
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    print("=== 数値列の統計量 ===")
    print(df[numeric_cols].describe().to_string())
else:
    print("数値列なし")

=== 数値列の統計量 ===
            Valence       Arousal     Dominance
count  10039.000000  10039.000000  10039.000000
mean       2.778066      3.089750      3.194326
std        0.897311      0.701749      0.789100
min        1.000000      1.000000      0.500000
25%        2.000000      2.500000      2.500000
50%        2.500000      3.000000      3.000000
75%        3.500000      3.500000      4.000000
max        5.500000      5.000000      5.000000


In [ ]:
# Cell 6: カテゴリ列の分布を見やすく表示
import math

cat_cols = df.select_dtypes(include=[object]).columns.tolist()
# ユニーク値が多すぎる列（IDやファイル名系）は除外
plot_cols = [c for c in cat_cols if df[c].nunique() <= 30]

n = len(plot_cols)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows))
axes = axes.flatten() if n > 1 else [axes]

for ax, col in zip(axes, plot_cols):
    vc = df[col].value_counts()
    colors = plt.cm.tab10.colors[:len(vc)]
    bars = ax.barh(vc.index.astype(str)[::-1], vc.values[::-1], color=colors[::-1])
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel("件数", fontsize=10)
    # 各バーに件数ラベルを付ける
    for bar, v in zip(bars, vc.values[::-1]):
        ax.text(bar.get_width() + vc.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{v:,}', va='center', fontsize=9)
    ax.set_xlim(0, vc.max() * 1.18)
    ax.spines[['top', 'right']].set_visible(False)

# 余ったサブプロットを非表示
for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout(pad=2.0)
plt.show()

In [7]:
# Cell 7: ファイルパス列の調査（音声データ用）
# パスらしき列を自動検出
path_like_cols = [c for c in df.columns if any(k in c.lower() for k in ['file', 'path', 'wav', 'audio'])]
print(f"パス系列の候補: {path_like_cols}")

for col in path_like_cols:
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
    print(f"\n[{col}] サンプル値: {sample}")
    # 絶対パスか相対パスか
    from pathlib import PurePath
    if sample:
        p = PurePath(str(sample))
        print(f"  絶対パス判定: {p.is_absolute()}")
        print(f"  拡張子: {p.suffix}")

パス系列の候補: ['filename']

[filename] サンプル値: Ses01F_impro01_F000.wav
  絶対パス判定: False
  拡張子: .wav
